# IS-Bench: prepare data for the sink task

Connect this notebook to the **same Colab runtime** that completed `isbench_colab_setup.ipynb`. Run cells in order. A missing `/content/miniforge3/envs/isbench-gpu/bin/python` means that runtime was reset and setup needs to run again.

This stage installs the BEHAVIOR scene/object data needed by OmniGibson 1.1.1, checks out the IS-Bench code at the commit used in our offline experiment, and extracts the authors' sink-with-blender scene. The BEHAVIOR archive is approximately 22.3 GB compressed; the IS-Bench scene archive is about 49 MB. Allow additional disk space for extraction. The storage check below uses a conservative 100 GiB reserve before the BEHAVIOR download.

Subprocess output is explicitly streamed into the notebook so that the Antigravity Colab extension saves it. This stage does not start the simulator. After `SCENE_DATA_READY`, the next step is to execute the authors' example plan for this one task before adding the VLM.

Sources: [IS-Bench instructions](https://github.com/AI45Lab/IS-Bench), [OmniGibson 1.1.1 downloader](https://github.com/StanfordVL/OmniGibson/blob/v1.1.1/omnigibson/utils/asset_utils.py).


## 1. Reuse the installed environment and show its output

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

python = Path("/content/miniforge3/envs/isbench-gpu/bin/python")
assert python.is_file(), "Reconnect to the existing Colab runtime, or run the setup notebook first."

def run(*args, env=None):
    args = list(map(str, args))
    print("Running:", " ".join(args[:6]), flush=True)
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          stdin=subprocess.DEVNULL, text=True, bufsize=1, env=env) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status != 0:
        raise subprocess.CalledProcessError(status, args)

data_env = os.environ.copy()
data_env["OMNIGIBSON_NO_OMNIVERSE"] = "1"
data_env["PYTHONUNBUFFERED"] = "1"
run(python, "-c", "import torch; from importlib.metadata import version; "
    "assert torch.cuda.is_available(); assert version('omnigibson') == '1.1.1'; "
    "print('GPU:', torch.cuda.get_device_name(0)); print('PyTorch:', torch.__version__); "
    "print('OmniGibson:', version('omnigibson')); print('Isaac Sim:', version('isaacsim'))")
run("df", "-h", "/content")

query = "from omnigibson.macros import gm; import json; print(json.dumps({'dataset': gm.DATASET_PATH, 'assets': gm.ASSET_PATH, 'key': gm.KEY_PATH}))"
paths = json.loads(subprocess.check_output([str(python), "-c", query], env=data_env, text=True))
dataset_path = Path(paths["dataset"])
print("BEHAVIOR dataset destination:", dataset_path)


## 2. Read the dataset terms, then download the scene/object data

In [ ]:
run(python, "-c", "from omnigibson.utils.asset_utils import print_user_agreement; print_user_agreement()",
    env=data_env)


In [ ]:
# agreement = input("After reading the terms above, type yes if you agree: ").strip().lower()
# if agreement != "yes":
#     raise RuntimeError("Dataset download was not started.")

metadata = dataset_path / "metadata/avg_category_specs.json"
if dataset_path.exists() and not metadata.is_file():
    raise RuntimeError(
        f"An incomplete dataset directory already exists at {dataset_path}. "
        "Stop here and report this message before retrying."
    )
if not metadata.is_file():
    free_gib = shutil.disk_usage("/content").free / (1024 ** 3)
    print(f"Available disk: {free_gib:.1f} GiB")
    if free_gib < 100:
        raise RuntimeError("This full-dataset download needs more disk space; report the available disk before proceeding.")

download = r'''import time
from omnigibson.utils import asset_utils

last_update = [0.0]
def progress(block, block_size, total):
    downloaded = min(block * block_size, total) if total > 0 else block * block_size
    now = time.monotonic()
    if now - last_update[0] >= 10 or (total > 0 and downloaded >= total):
        last_update[0] = now
        print(f"Downloaded {downloaded / 1e9:.2f} / {total / 1e9:.2f} GB", flush=True)

asset_utils.show_progress = progress
asset_utils.download_og_dataset(accept_license=True)
print("BEHAVIOR_DOWNLOAD_FINISHED", flush=True)'''
run(python, "-c", download, env=data_env)
assert metadata.is_file(), "Dataset metadata is missing after the download."
assert (dataset_path / "scenes/Rs_int").is_dir(), "Rs_int base scene is missing."
assert (dataset_path / "objects/blender").is_dir(), "Blender objects are missing."
assert Path(paths["key"]).is_file(), "Dataset key was not installed."

# The official downloader retains this temporary archive after extraction.
archive = Path("/tmp/og_dataset.tar.gz")
if archive.is_file():
    archive.unlink()
print("BEHAVIOR_DATA_READY")
run("df", "-h", "/content")


## 3. Get the same IS-Bench code revision used in the offline trial

In [ ]:
repo = Path("/content/IS-Bench")
commit = "6a406e162feca1c4c455d9c9f1f58832edbdc1fd"
if not repo.exists():
    run("git", "clone", "https://github.com/AI45Lab/IS-Bench.git", repo)
    run("git", "-C", repo, "checkout", "--detach", commit)
current = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
assert current == commit, f"Existing repository is at {current}, expected {commit}. Stop before changing it."
task_name = "clean_a_kitchen_sink__with_blender"
scene = "Rs_int"
task = json.loads((repo / "data/tasks" / f"{task_name}.json").read_text())
assert task["scene_info"]["default_scene_model"] == scene
print("Repository commit:", current)
print("Task:", task["planning_context"]["task_instruction"])


## 4. Download and verify the authors' scene archive, then extract the sink scene

In [ ]:
import hashlib
import tarfile
from urllib.request import urlretrieve

scene_archive = Path("/content/isbench_scenes.tar.gz")
expected_hash = "0552f5108db89bbfd4e9f5b046634b64c61c1aa48c13d52c638fc1c47cf6296d"
if not scene_archive.exists():
    print("Downloading IS-Bench scene archive...", flush=True)
    urlretrieve("https://huggingface.co/datasets/Ursulalala/IS_Bench_scenes/resolve/main/scenes.tar.gz", scene_archive)
digest = hashlib.sha256(scene_archive.read_bytes()).hexdigest()
assert digest == expected_hash, "Scene archive changed or is incomplete. Stop and report this message."

relative_scene = f"scenes/{scene}/json/{scene}_task_{task_name}_0_0_template.json"
scene_file = repo / "data" / relative_scene
with tarfile.open(scene_archive, "r:gz") as archive:
    member = archive.getmember(relative_scene)
    assert member.isfile(), "Expected scene JSON was not a regular file."
    content = archive.extractfile(member).read()
scene_data = json.loads(content)
scene_file.parent.mkdir(parents=True, exist_ok=True)
scene_file.write_bytes(content)
print("Saved sink scene:", scene_file)


## 5. Check that the scene's referenced object models exist

In [ ]:
missing = []
checked = 0
for item in scene_data["objects_info"]["init_info"].values():
    if item.get("class_name") != "DatasetObject":
        continue
    args = item["args"]
    model_dir = dataset_path / "objects" / args["category"] / args["model"]
    checked += 1
    if not model_dir.is_dir() or not any(model_dir.rglob("*.encrypted.usd")):
        missing.append(str(model_dir))
assert not missing, "Missing object models: " + "\n".join(missing[:10])

manifest = {
    "task": task_name, "scene": scene, "repo_commit": current,
    "scene_archive_sha256": digest, "scene_file": str(scene_file),
    "dataset_path": str(dataset_path), "object_references_checked": checked,
    "scope": "data preparation; scene execution has not been tested in this notebook",
}
manifest_path = Path("/content/isbench_scene_data_manifest.json")
manifest_path.write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print("SCENE_DATA_READY")
